In [ ]:
# 1. Dependencies & Setup
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.style.use('seaborn-v0_8-darkgrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
# 2. Objective Function
def rastrigin_2d(x):
    x = x.to(dtype=torch.float32)
    return 20.0 + x[..., 0] ** 2 - 10.0 * torch.cos(2.0 * torch.pi * x[..., 0]) + x[..., 1] ** 2 - 10.0 * torch.cos(2.0 * torch.pi * x[..., 1])

In [ ]:
# 3. PyTorch PSO Engine
class CustomPSO:
    def __init__(
        self,
        objective_fn,
        n_particles=40,
        dim=2,
        bounds=[[-5.12, 5.12], [-5.12, 5.12]],
        n_iter=80,
        inertia=0.7,
        c1=1.5,
        c2=1.5,
        seed=7,
        device=None,
    ):
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.device = device or torch.device('cpu')
        self.objective_fn = objective_fn
        self.n_particles = n_particles
        self.dim = dim
        self.n_iter = n_iter
        self.inertia = inertia
        self.c1 = c1
        self.c2 = c2

        self.lower = torch.tensor([b[0] for b in bounds], dtype=torch.float32, device=self.device)
        self.upper = torch.tensor([b[1] for b in bounds], dtype=torch.float32, device=self.device)

        self.positions = torch.rand(n_particles, dim, device=self.device) * (self.upper - self.lower) + self.lower
        self.velocities = torch.rand(n_particles, dim, device=self.device) * 2.0 - 1.0

        self.personal_best_positions = self.positions.clone()
        self.personal_best_scores = self.objective_fn(self.positions)

        self.global_best_score = self.personal_best_scores.min().clone()
        self.global_best_position = self.personal_best_positions[torch.argmin(self.personal_best_scores)].clone()

        self.global_best_history = []
        self.position_history = []

    def _clip_positions(self):
        self.positions = torch.where(
            self.positions < self.lower,
            self.lower,
            torch.where(self.positions > self.upper, self.upper, self.positions),
        )

    def run(self):
        self.global_best_history = []
        self.position_history = []

        for _ in range(self.n_iter):
            with torch.no_grad():
                scores = self.objective_fn(self.positions)

                personal_best_mask = scores < self.personal_best_scores
                self.personal_best_positions = torch.where(
                    personal_best_mask[:, None],
                    self.positions,
                    self.personal_best_positions,
                )
                self.personal_best_scores = torch.where(personal_best_mask, scores, self.personal_best_scores)

                global_best_mask = self.personal_best_scores < self.global_best_score
                masked_positions = torch.where(
                    global_best_mask[:, None],
                    self.personal_best_positions,
                    torch.full_like(self.personal_best_positions, torch.inf),
                )
                masked_scores = torch.where(
                    global_best_mask,
                    self.personal_best_scores,
                    torch.full_like(self.personal_best_scores, torch.inf),
                )

                if global_best_mask.any().item():
                    best_index = torch.argmin(masked_scores)
                    self.global_best_position = masked_positions[best_index].clone()
                    self.global_best_score = masked_scores[best_index].clone()

                r1 = torch.rand(self.n_particles, self.dim, device=self.device)
                r2 = torch.rand(self.n_particles, self.dim, device=self.device)

                self.velocities = (
                    self.inertia * self.velocities
                    + self.c1 * r1 * (self.personal_best_positions - self.positions)
                    + self.c2 * r2 * (self.global_best_position.unsqueeze(0).expand_as(self.positions) - self.positions)
                )
                self.positions = self.positions + self.velocities
                self._clip_positions()

                self.global_best_history.append(self.global_best_score.detach().cpu().item())
                self.position_history.append(self.positions.detach().cpu().numpy().copy())

        return {
            'global_best_history': self.global_best_history,
            'particle_positions': self.position_history,
            'best_position': self.global_best_position.detach().cpu().numpy(),
        }

In [ ]:
# 4. Execution and Convergence Chart
pso = CustomPSO(
    objective_fn=rastrigin_2d,
    n_particles=35,
    dim=2,
    bounds=[[-5.12, 5.12], [-5.12, 5.12]],
    n_iter=80,
    device=device,
)
history = pso.run()

plt.figure(figsize=(8, 4))
plt.plot(history['global_best_history'], linewidth=2, color='royalblue')
plt.xlabel('Iteration')
plt.ylabel('Global Best Fitness')
plt.title('PSO Convergence Curve')
plt.grid(True, alpha=0.3)
plt.show()

print(f'Best position found: {history["best_position"]}')
print(f'Best fitness: {history["global_best_history"][-1]:.4f}')

In [ ]:
# 5. 2D Swarm Simulation
x_vals = torch.linspace(-5.12, 5.12, 300, device=device)
y_vals = torch.linspace(-5.12, 5.12, 300, device=device)
X, Y = torch.meshgrid(x_vals, y_vals, indexing='ij')
Z = rastrigin_2d(torch.stack([X, Y], dim=-1))

fig, ax = plt.subplots(figsize=(7, 7))
contour = ax.contourf(X.cpu().numpy(), Y.cpu().numpy(), Z.cpu().numpy(), levels=40, cmap='viridis')
scatter = ax.scatter([], [], s=40, c='red', alpha=0.8)
ax.set_xlim(-5.12, 5.12)
ax.set_ylim(-5.12, 5.12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Particle Swarm Optimization in 2D')

def update(frame):
    positions = history['particle_positions'][frame]
    scatter.set_offsets(positions)
    return scatter,

anim = FuncAnimation(fig, update, frames=range(len(history['particle_positions'])), interval=60, blit=True)
HTML(anim.to_html5_video())